# Problem 027:  Quadratic Primes

> Euler discovered the remarkable quadratic formula:
> $$n^2 + n + 41$$
> It turns out that the formula will produce $40$ primes for the consecutive integer values $0 \le n \le 39$. However, when $n = 40, 40^2 + 40 + 41 = 40(40 + 1) + 41$ is divisible by $41$, and certainly when $n = 41, 41^2 + 41 + 41$ is clearly divisible by $41$.
>
> The incredible formula $n^2 - 79n + 1601$ was discovered, which produces $80$ primes for the consecutive values $0 \le n \le 79$. The product of the coefficients, $-79$ and $1601$, is $-126479$.
>
> Considering quadratics of the form:
>
>>  $n^2 + an + b$    , where $|a| < 1000$ and $|b| < 1000$ <br>
>>  where $|n|$ is the modulus/absolute value of $n$. <br>
>>  e.g. $|11|=11$ and $|-4|=4$
>
> Find the product of the coefficients, $a$ and $b$, for the quadratic expression that produces the maximum number of primes for consecutive values of $n$, starting with $n=0$.


## Solution $\mathcal O\left(\frac{N^2}{\ln N}\right)$

Without any insight in what makes a polynomial produce primes or not, a brute force search is done.  That said, lots of possible combonations can be discarded.

For one, $b$ must be prime.  This can be seen in that the first value in the polynomial will be $f(0) = b$. This restriction reduces the search over $b$ from $2N$ to approximately $N / \ln N$.

Next, $a$ will be odd.  If $a$ were to be even, then $f(2x+1)$ will be even for all values of $x$, thus restricting the number of primes generated.  Yes, one of them could be $2$, but that can happen a maximum of two times for a parabola.  This halves the search space in $a$.

Also, $a > -b$.  This restriction ensures that $f(1) = 1 + a + b > 1$.  This grants a further factor of two in the search space of $a$.

Regardless of the restrictions placed on the search, I wrote the algorithm `count_polynomial_primes` to account for any integer input of $a$ and $b$.  This requires some unnecessary checks in the front of the code, but feels a little more robust than just writing a code for a restricted set of inputs.

In [1]:
%%time

def extend_sieve(pow2: int, primes: list[int]) -> list[int]:
    s = [True] * pow2
    # seive previous primes
    for p in primes[1:]:
        if p * p > 4*pow2:
            break
        i0 = ((2 * pow2) // p + 1) * p
        if i0 % 2 == 0:
            i0 += p
        i0 = (i0 - 2*pow2) // 2
        l = (pow2 - i0-1) // p + 1
        s[i0::p] = [False] * l
    # search for new primes
    primes += [2*(pow2+i)+1 for i in range(pow2) if s[i]]
    return s

def count_polynomial_primes(a: int, b: int, pow2: int, s: list[int], primes: list[int]) -> (int, int):
    if b < 2:
        return 0

    if b == 2:
        f = 3 + a
        if f == 2:
            return 2
        while f > 2 * pow2:
            s += extend_sieve(pow2, primes)
            pow2 *= 2
        if f % 2 and f > 0 and s[f//2]:
            return 2
        return 1

    if b % 2 == 0:
        return 0

    if not s[b//2]:
        return 0

    nprimes = 1
    f = 1 + a + b

    if f == 2:
        f = 2 * (2 + a) + b
        while f > 2 * pow2:
            s += extend_sieve(pow2, primes)
            pow2 *= 2        
        if f % 2 and f > 0 and s[f//2]:
            return 3
        else:
            return 2
    
    while f > 2 * pow2:
        s += extend_sieve(pow2, primes)
        pow2 *= 2
    while f % 2 and f > 0 and s[f//2]:
        nprimes += 1
        f = nprimes * (nprimes + a) + b
        while f > 2 * pow2:
            s += extend_sieve(pow2, primes)
            pow2 *= 2
    return nprimes, pow2

def p027(N: int) -> int:
    # initialize primes up to `N`
    pow2 = 2
    primes = [2,3]
    s = [False, True]
    while 2 * pow2 < N:
        s += extend_sieve(pow2, primes)
        pow2 *= 2

    val_max = (0, 0)
    amax = 2 * (N // 2) + 1
    nb = 1
    b = 3
    while b < N:
        amin = 2 - b
        for a in range(amin, amax, 2):
            nprimes, pow2 = count_polynomial_primes(a, b, pow2, s, primes)
            val_max = max(val_max, (nprimes, a * b))
        nb += 1
        b = primes[nb]
    return val_max[1]

N = 1_000
ans = p027(N)

print(ans)

-59231
CPU times: user 175 ms, sys: 330 μs, total: 175 ms
Wall time: 187 ms


This problem is a little underwhelming in the end.  Ultimately, the polynomial $n^2+n+41$ is rather unique in its prime generating properties.  Notice how this polynomial has a maximum at $-1/2$, and is symmetric about this point.  Thus the polynomial $(n-c)^2 + (n-c) + 41$ will produce $40+c$ primes in a row when $|c|<41$, even if there will be duplicates.  In fact, the other polynomial listed in the problem is $n^2 - 79n + 1601 = (n-40)^2 + (x - 40) + 41$.

If one just presumes that this is the only class of polynomials that generate the most primes in the desired range, then:
$$(n-c)^2 + (n-c) + 41 = n^2 + (1-2c)n +(c^2-c+41)$$
The restrictions on $a$ and $b$ make it so we are looking for the largest $c$ s.t. $c^2-c+41 < N$:
$$c < \frac{1}{2} \left(1 + \sqrt{4N-163}\right)$$

For $N=1000$ we find: $c = 31$, $a = -61$, $b = 971$, and $ab = -59231$.

Now _a priori_ we don't have the insight that this one class of polynomials will produce the answer, so a brute force search is the more appropriate way of finding the answer, regardless of how trivial the actual solution is.